# Jena Weather - Full Pipeline Training

Demonstrates the complete MLOps workflow using all noted tools:
- **Hydra** - load config from project config/ directory
- **DVC** - track training data with version control
- **MLflow** - track experiments, log metrics, register model

**Usage for Snapshot Testing:**
1. Run All cells to produce a FINISHED MLflow run
2. Navigate to Experiments > find the run
3. Click 'Create Snapshot' to test the snapshot workflow

In [1]:
import numpy as np
import pandas as pd
import os

# Generate synthetic Jena-style weather data
np.random.seed(42)
n_samples = 1000
days = np.arange(n_samples)

temperature = 10 + 15 * np.sin(2 * np.pi * days / 365) + np.random.normal(0, 3, n_samples)
humidity = 60 + 20 * np.cos(2 * np.pi * days / 365) + np.random.normal(0, 5, n_samples)
pressure = 1013 + 10 * np.sin(2 * np.pi * days / 180) + np.random.normal(0, 2, n_samples)
wind_speed = 5 + 3 * np.abs(np.sin(2 * np.pi * days / 90)) + np.random.normal(0, 1.5, n_samples)

df = pd.DataFrame({
    'temperature': temperature,
    'humidity': humidity,
    'pressure': pressure,
    'wind_speed': wind_speed,
    'target': np.roll(temperature, -1),
})
df = df.iloc[:-1]

# Save dataset
os.makedirs('data', exist_ok=True)
df.to_csv('data/dataset.csv', index=False)
print(f'Dataset saved: {len(df)} rows, {len(df.columns)} columns')
print(f'Columns: {list(df.columns)}')

Dataset saved: 999 rows, 5 columns
Columns: ['temperature', 'humidity', 'pressure', 'wind_speed', 'target']


In [2]:
import subprocess

# Track dataset with DVC
try:
    result = subprocess.run(
        ['dvc', 'add', 'data/dataset.csv'],
        capture_output=True, text=True, cwd=os.getcwd()
    )
    if result.returncode == 0:
        print('DVC: data/dataset.csv tracked successfully')
        print(f'  .dvc file created: data/dataset.csv.dvc')
    else:
        print(f'DVC tracking note: {result.stderr.strip()}')
except FileNotFoundError:
    print('DVC not installed in this environment - skipping data tracking')

DVC: data/dataset.csv tracked successfully
  .dvc file created: data/dataset.csv.dvc


In [3]:
import yaml

# Load Hydra config (injected by noted as __noted_hydra_config__)
# Falls back to manual YAML load if config selector is not active
try:
    config = __noted_hydra_config__
    print('Config loaded from noted Hydra injection')
except NameError:
    with open('config/config.yaml') as f:
        config = yaml.safe_load(f)
    # Load model sub-config
    defaults = config.get('defaults', [])
    model_name = 'linear'
    for d in defaults:
        if isinstance(d, dict) and 'model' in d:
            model_name = d['model']
    model_path = f'config/model/{model_name}.yaml'
    if os.path.exists(model_path):
        with open(model_path) as f:
            config['model'] = yaml.safe_load(f)
    print(f'Config loaded from YAML files (model: {model_name})')

training_cfg = config.get('training', {})
model_cfg = config.get('model', {})
print(f"Training: epochs={training_cfg.get('epochs')}, batch_size={training_cfg.get('batch_size')}, lr={training_cfg.get('learning_rate')}")
print(f"Model: type={model_cfg.get('type')}, params={model_cfg.get('params', {})}")

Config loaded from noted Hydra injection
Training: epochs=50, batch_size=32, lr=0.001
Model: type=None, params={}


In [4]:
import mlflow
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Prepare data
features = df.drop('target', axis=1)
target = df['target']
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=training_cfg.get('seed', 42)
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Select model from Hydra config
model_type = model_cfg.get('type', 'LinearRegression')
if 'Ridge' in model_type:
    model = Ridge(alpha=1.0)
else:
    model = LinearRegression()

EXPERIMENT_NAME = 'jena_weather_regression'
epochs = training_cfg.get('epochs', 10)
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name=f'{model_type}_run'):
    # Log Hydra config as params
    mlflow.log_param('model_type', model_type)
    mlflow.log_param('epochs', epochs)
    mlflow.log_param('batch_size', training_cfg.get('batch_size', 32))
    mlflow.log_param('learning_rate', training_cfg.get('learning_rate', 0.001))
    mlflow.log_param('seed', training_cfg.get('seed', 42))
    mlflow.log_param('n_train', len(X_train))
    mlflow.log_param('n_test', len(X_test))
    mlflow.log_param('data_file', 'data/dataset.csv')
    mlflow.log_param('data_rows', len(df))
    for k, v in model_cfg.get('params', {}).items():
        mlflow.log_param(f'model_{k}', v)

    # Train
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)

    # Metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)

    r2 = r2_score(y_test, y_pred)

    # Log step metrics (simulates epoch training)
    mlflow.log_metric('total_epochs', epochs)
    for epoch in range(epochs):
        progress = (epoch + 1) / epochs
        mlflow.log_metric('mae', mae + (1 - progress) * 0.5, step=epoch)
        mlflow.log_metric('rmse', rmse + (1 - progress) * 0.8, step=epoch)
        mlflow.log_metric('r2', r2 - (1 - progress) * 0.05, step=epoch)
        mlflow.log_metric('epoch', epoch + 1, step=epoch)

    # Final metrics
    mlflow.log_metric('mae', mae)
    mlflow.log_metric('rmse', rmse)
    mlflow.log_metric('r2', r2)

    # Log model artifact
    mlflow.sklearn.log_model(model, name="model")

    run_id = mlflow.active_run().info.run_id
    print(f'Model: {model_type}')
    print(f'MAE:  {mae:.4f}')
    print(f'RMSE: {rmse:.4f}')
    print(f'R2:   {r2:.4f}')
    print(f'Epochs: {epochs}')
    print(f'Run ID: {run_id}')
    print(f'Experiment: {EXPERIMENT_NAME}')

2026/03/23 23:17:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Model: LinearRegression
MAE:  3.2080
RMSE: 3.9523
R2:   0.8688
Epochs: 50
Run ID: a6f123b9c5d04664a4226795078137f7
Experiment: jena_weather_regression
🏃 View run LinearRegression_run at: http://mlflow:5000/#/experiments/146/runs/a6f123b9c5d04664a4226795078137f7
🧪 View experiment at: http://mlflow:5000/#/experiments/146


## What was captured

| Tool | What | Details |
|------|------|--------|
| **DVC** | Data version | data/dataset.csv.dvc tracks the training data |
| **Hydra** | Config | Training params from config/config.yaml + model group |
| **MLflow** | Experiment run | Params, metrics (MAE/RMSE/R2), sklearn model artifact |
| **Git** | Code version | Current commit captures notebook + config state |

## Snapshot Testing

1. Go to **Experiments** > jena_weather_regression > click the run
2. Click **Create Snapshot** - captures git + DVC + Hydra + MLflow state
3. Modify something (e.g., change model config to gru) and re-run
4. **Restore Snapshot** returns to the exact state when snapshot was taken
5. **New Experiment from Snapshot** forks a new branch for experimentation